<a href="https://colab.research.google.com/github/MarcinKuriata/cyclistic-bike-share-analysis/blob/main/01_cyclistic_exploratory_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#Library import to use functions
import pandas as pd
import numpy as np

# Load quarterly datasets from Google Drive(connected with Google Colab)
q1 = pd.read_csv('/content/drive/MyDrive/Cyclistic_capstone/Divvy_Trips_2019_Q1.csv')
q2 = pd.read_csv('/content/drive/MyDrive/Cyclistic_capstone/Divvy_Trips_2019_Q2.csv')
q3 = pd.read_csv('/content/drive/MyDrive/Cyclistic_capstone/Divvy_Trips_2019_Q3.csv')
q4 = pd.read_csv('/content/drive/MyDrive/Cyclistic_capstone/Divvy_Trips_2019_Q4.csv')

# Verify column consistency across all datasets
print('Q1 columns:', q1.columns.tolist())
print('Q2 columns:', q2.columns.tolist())
print('Q3 columns:', q3.columns.tolist())
print('Q4 columns:', q4.columns.tolist())

Q1 columns: ['trip_id', 'start_time', 'end_time', 'bikeid', 'tripduration', 'from_station_id', 'from_station_name', 'to_station_id', 'to_station_name', 'usertype', 'gender', 'birthyear']
Q2 columns: ['01 - Rental Details Rental ID', '01 - Rental Details Local Start Time', '01 - Rental Details Local End Time', '01 - Rental Details Bike ID', '01 - Rental Details Duration In Seconds Uncapped', '03 - Rental Start Station ID', '03 - Rental Start Station Name', '02 - Rental End Station ID', '02 - Rental End Station Name', 'User Type', 'Member Gender', '05 - Member Details Member Birthday Year']
Q3 columns: ['trip_id', 'start_time', 'end_time', 'bikeid', 'tripduration', 'from_station_id', 'from_station_name', 'to_station_id', 'to_station_name', 'usertype', 'gender', 'birthyear']
Q4 columns: ['trip_id', 'start_time', 'end_time', 'bikeid', 'tripduration', 'from_station_id', 'from_station_name', 'to_station_id', 'to_station_name', 'usertype', 'gender', 'birthyear']


In [6]:
# Rename Q2 columns to match Q1, Q3, and Q4 schema
q2 = q2.rename(columns={
    '01 - Rental Details Rental ID': 'trip_id',
    '01 - Rental Details Local Start Time': 'start_time',
    '01 - Rental Details Local End Time': 'end_time',
    '01 - Rental Details Bike ID': 'bikeid',
    '01 - Rental Details Duration In Seconds Uncapped': 'tripduration',
    '03 - Rental Start Station ID': 'from_station_id',
    '03 - Rental Start Station Name': 'from_station_name',
    '02 - Rental End Station ID': 'to_station_id',
    '02 - Rental End Station Name': 'to_station_name',
    'User Type': 'usertype',
    'Member Gender': 'gender',
    '05 - Member Details Member Birthday Year': 'birthyear'
})

# Verify that Q2 columns now match Q1
print("Q2 columns fixed:", q2.columns.tolist() == q1.columns.tolist() == q3.columns.tolist() == q4.columns.tolist())

Q2 columns fixed: True


In [15]:
# Combine all quarters into a single DataFrame
df_2019 = pd.concat([q1, q2, q3, q4],          #could also use UNION ALL in SQL
                    ignore_index=True)         #remove old, separate indexing to give new indexing for dataset as a whole

# Inspect dataset dimensions and structure
print(f"Total records: {df_2019.shape[0]:,}")  #show number of rows in coma-separated format
print(f"Total columns: {df_2019.shape[1]}")    #show number of columns in coma-separated format

Total records: 3,818,004
Total columns: 12


In [16]:
# Inspect first 3 rows of a DataFrame
df_2019.head(3)

,trip_id,start_time,end_time,bikeid,tripduration,from_station_id,from_station_name,to_station_id,to_station_name,usertype,gender,birthyear
0,21742443,2019-01-01 00:04:37,2019-01-01 00:11:07,2167,390.0,199,Wabash Ave & Grand Ave,84,Milwaukee Ave & Grand Ave,Subscriber,Male,1989.0
1,21742444,2019-01-01 00:08:13,2019-01-01 00:15:34,4386,441.0,44,State St & Randolph St,624,Dearborn St & Van Buren St (*),Subscriber,Female,1990.0
2,21742445,2019-01-01 00:13:23,2019-01-01 00:27:12,1524,829.0,15,Racine Ave & 18th St,644,Western Ave & Fillmore St (*),Subscriber,Female,1994.0


In [17]:
#Show Dataframe info
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3818004 entries, 0 to 3818003
Data columns (total 12 columns):
 #   Column             Dtype  
---  ------             -----  
 0   trip_id            int64  
 1   start_time         object 
 2   end_time           object 
 3   bikeid             int64  
 4   tripduration       object 
 5   from_station_id    int64  
 6   from_station_name  object 
 7   to_station_id      int64  
 8   to_station_name    object 
 9   usertype           object 
 10  gender             object 
 11  birthyear          float64
dtypes: float64(1), int64(4), object(7)
memory usage: 349.5+ MB


### 🔍 Initial Data Quality Observations & Next Steps:
* **Schema Inconsistencies:** Column names across `Q2` differed from `Q1`, `Q3`, and `Q4` and were standardized prior to unioning.
* **Incorrect Data Types:**
  * `start_time` and `end_time` are currently stored as `object` (string) instead of `TIMESTAMP`/`DATETIME`.
  * `tripduration` is stored as `object` (string) instead of numeric (`FLOAT`/`INTEGER`).
* **Missing Values:** `gender` and `birthyear` contain null values.
* **Architectural Decision:** To maintain an efficient **ELT (Extract, Load, Transform)** workflow and demonstrate robust data-warehouse capabilities, full data cleansing, type casting, outlier removal, and feature engineering will be handled directly in **Google BigQuery using SQL**.

In [18]:
# Export raw consolidated dataset without additional index column
df_2019.to_csv('/content/drive/MyDrive/Cyclistic_capstone/divvy_trips_2019_raw.csv', index=False)